In [1]:
!pip install transformers sentencepiece

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Multi-language translation model loaded successfully!")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Multi-language translation model loaded successfully!


In [3]:
text = "I am a student."

# Set source language: English
tokenizer.src_lang = "eng_Latn"

# Translate to Urdu
inputs = tokenizer(text, return_tensors="pt")

outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids("urd_Arab")
)

translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("English:", text)
print("Urdu:", translation)

English: I am a student.
Urdu: میں طالب علم ہوں.


In [4]:
languages = {
    "English": "eng_Latn",
    "Urdu": "urd_Arab",
    "French": "fra_Latn",
    "Spanish": "spa_Latn",
    "German": "deu_Latn",
    "Arabic": "arb_Arab",
    "Chinese": "zho_Hans",
    "Hindi": "hin_Deva",
    "Russian": "rus_Cyrl",
    "Portuguese": "por_Latn",
    "Italian": "ita_Latn",
    "Japanese": "jpn_Jpan",
    "Korean": "kor_Hang"
}

print("Languages available:")
for language in languages:
    print("-", language)

Languages available:
- English
- Urdu
- French
- Spanish
- German
- Arabic
- Chinese
- Hindi
- Russian
- Portuguese
- Italian
- Japanese
- Korean


In [5]:
source_language = input("Enter source language: ")
target_language = input("Enter target language: ")
text = input("Enter text to translate: ")

# Get language codes
source_code = languages[source_language]
target_code = languages[target_language]

# Set source language
tokenizer.src_lang = source_code

# Tokenize input
inputs = tokenizer(text, return_tensors="pt")

# Generate translation
outputs = model.generate(
    **inputs,
    forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code)
)

# Decode translation
translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("\nSource:", source_language)
print("Text:", text)
print("Target:", target_language)
print("Translation:", translation)

Enter source language:  English
Enter target language:  Urdu
Enter text to translate:  good



Source: English
Text: good
Target: Urdu
Translation: اچھا


In [6]:
!pip install gradio

In [7]:
import gradio as gr


# Translation function
def translate_text(text, source_language, target_language):
    if not text.strip():
        return ""

    source_code = languages[source_language]
    target_code = languages[target_language]

    tokenizer.src_lang = source_code

    inputs = tokenizer(text, return_tensors="pt")

    outputs = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(target_code)
    )

    translation = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return translation


# Swap languages
def swap_languages(source, target, text):
    return target, source, text


# Supported languages
language_list = list(languages.keys())


# Create professional interface
with gr.Blocks(title="AI Language Translator") as app:

    gr.Markdown(
        """
        # 🌍 AI Language Translator
        ### Translate text between multiple languages using Artificial Intelligence

        Enter your text, select the source and target languages, and click **Translate**.
        """
    )

    with gr.Row():

        source_dropdown = gr.Dropdown(
            choices=language_list,
            value="English",
            label="🔤 From"
        )

        target_dropdown = gr.Dropdown(
            choices=language_list,
            value="Urdu",
            label="🌐 To"
        )

    input_text = gr.Textbox(
        label="📝 Enter Text",
        placeholder="Type your text here...",
        lines=5
    )

    with gr.Row():

        translate_button = gr.Button(
            "🔄 Translate",
            variant="primary"
        )

        swap_button = gr.Button(
            "↔️ Swap Languages"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )

    output_text = gr.Textbox(
        label="✨ Translation",
        lines=5
    )


    # Translate button
    translate_button.click(
        fn=translate_text,
        inputs=[
            input_text,
            source_dropdown,
            target_dropdown
        ],
        outputs=output_text
    )


    # Swap button
    swap_button.click(
        fn=swap_languages,
        inputs=[
            source_dropdown,
            target_dropdown,
            input_text
        ],
        outputs=[
            source_dropdown,
            target_dropdown,
            input_text
        ]
    )


    # Clear button
    clear_button.click(
        fn=lambda: ("", ""),
        inputs=[],
        outputs=[
            input_text,
            output_text
        ]
    )


# Launch application
app.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8125274465362b09c9.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 🌍 AI Language Translation Tool

## Project Overview

This project is an AI-powered language translation tool developed as part of the CodeAlpha AI Internship.

The application uses the **Facebook NLLB-200 multilingual translation model** from Hugging Face Transformers to translate text between multiple languages.

## Features

- 🌐 Supports multiple languages
- 🔄 Real-time text translation
- ↔️ Swap source and target languages
- 🗑️ Clear input and translation
- 🤖 Uses a pretrained AI translation model
- 💻 Simple and user-friendly Gradio interface

## Technologies Used

- Python
- Hugging Face Transformers
- NLLB-200
- PyTorch
- Gradio
- Kaggle Notebook

## Objective

The objective of this project is to develop a simple and accessible AI-based translation application that allows users to translate text between different languages through an easy-to-use interface.

## Conclusion

The AI Language Translation Tool successfully provides multilingual text translation through a Gradio web interface powered by the NLLB-200 AI model.